# Effect of cuts on categorization

In [1]:
import pandas as pd
import ROOT
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
plt.rcParams["figure.figsize"] = (20,10)
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
pd.set_option("max_colwidth", 1000)
pd.set_option("display.precision", 2)

# Loading from extracted file until we can get the data
rdfsep = ROOT.RDataFrame("DecayTree", "temp.root")
dfsep = pd.DataFrame(rdfsep.Cache().AsNumpy())
rdfsep.Count().GetValue()

Welcome to JupyROOT 6.28/00


1207697

In [2]:
# Adding the categories as string
import categories4 as f
dfsep['simplified_key'] = dfsep.apply(lambda row: f.pretty_categories_map[row["simplified"]], axis=1)
dfsep['key'] = dfsep.apply(lambda row: f.categories_map[row["category"]], axis=1)

In [3]:
def mygroupby(d, groupbycols):
    g = d.groupby(groupbycols).size().reset_index(name='count').sort_values([ 'count'], ascending=False).reset_index(drop=True)
    g["Percentage"] = g.apply(lambda row: 100 * row["count"]/d.shape[0], axis=1)
    g["cumulative %"] = g["Percentage"].cumsum(axis = 0)
    return g

In [4]:
def categ_groupby(df, groupbycol, threshold):
    """ Group the dataframe df by column groupbycol, grouping together entries rows with count < threshold """
    g = pd.DataFrame(df.groupby(groupbycol).count()["category"]).rename(columns={"category":"count"})
    g["Category"] = g.apply(lambda row: row.name if row["count"] > threshold else "others", axis=1)
    g2 = g.groupby("Category").sum()
    g2 = g2.sort_values([ 'count'], ascending=False)
    total = g2.sum()["count"]
    g2["Percentage"] = g2.apply(lambda row: 100 * row["count"]/total, axis=1)
    g2["cumulative %"] = g2["Percentage"].cumsum(axis = 0)
    return g, g2

# What was used for simplified categories...
# gs["Category"] = gs.apply(lambda row: row['simplified_key'] if row["count"] > 400 else "others", axis=1)
# gs2 = gs.groupby("Category").sum()
# gs2 = gs2.sort_values([ 'count'], ascending=False)
# total = gs2.sum()["count"]
# gs2["Percentage"] = gs2.apply(lambda row: 100 * row["count"]/total, axis=1)
# gs2["cumulative %"] = gs2["Percentage"].cumsum(axis = 0)

## Categorization with initial sample, 

It has the cuts
  - BDT_B
  - BDT_Y
  - BDT_Ds
  - BDT_Iso
  - PIDk
  
HOWEVER: the cut on q2_2 > 0 has been removed


In [18]:
_, g2 = categ_groupby(dfsep, "simplified_key", 400)
g2.style.format({'Percentage': "{:.2%}"}) 
print(g2[['count', 'Percentage']].to_markdown())
print(g2[['count', 'Percentage']].to_latex(float_format="%.2f" ))
#print(g2[['count', 'Percentage']].style.to_latex())

| Category           |   count |   Percentage |
|:-------------------|--------:|-------------:|
| Normalization like |  541243 |   44.8161    |
| Double Charm       |  464918 |   38.4962    |
| Bad Xc             |  183031 |   15.1554    |
| Signal             |    9966 |    0.825207  |
| Combinatorial      |    6657 |    0.551214  |
| Tau from charm     |    1441 |    0.119318  |
| others             |     441 |    0.0365158 |
\begin{tabular}{lrr}
\toprule
{} &   count &  Percentage \\
Category           &         &             \\
\midrule
Normalization like &  541243 &       44.82 \\
Double Charm       &  464918 &       38.50 \\
Bad Xc             &  183031 &       15.16 \\
Signal             &    9966 &        0.83 \\
Combinatorial      &    6657 &        0.55 \\
Tau from charm     &    1441 &        0.12 \\
others             &     441 &        0.04 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_266248/322890238.py:4: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(g2[['count', 'Percentage']].to_latex(float_format="%.2f" ))


In [19]:
_, g2 = categ_groupby(dfsep, "key", 400)
print(g2[['count', 'Percentage']].to_markdown())
print()
print(g2[['count', 'Percentage']].to_latex(float_format="%.2f", escape=True))

| Category                                         |   count |   Percentage |
|:-------------------------------------------------|--------:|-------------:|
| Xc_signal_Ypis_B_vertex_fromBs                   |  481611 |   39.8785    |
| Xc_background                                    |  183031 |   15.1554    |
| Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB   |  130954 |   10.8433    |
| Xc_signal_Ypis_displaced_fromBs_fromDs           |   84307 |    6.98081   |
| Xc_signal_Ypis_displaced_fromB0_fromDp           |   66975 |    5.54568   |
| Xc_signal_Ypis_nomatch_doubleCharm               |   40822 |    3.38015   |
| Xc_signal_Ypis_nomatch_Prompt                    |   37559 |    3.10997   |
| Xc_signal_Ypis_displaced_fromBp_fromD0           |   30159 |    2.49723   |
| Xc_signal_Ypis_displaced_fromLambdab_fromLambdac |   23554 |    1.95032   |
| Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB   |   23275 |    1.92722   |
| Xc_signal_Ypis_displaced_fromBs_fromDp           |   16769 |  

/tmp/ipykernel_266248/163666499.py:4: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(g2[['count', 'Percentage']].to_latex(float_format="%.2f", escape=True))


## Categorization with B_M < 5000

In [20]:
dfbmcut = dfsep.query("B_M < 5000")

In [21]:
_, g2 = categ_groupby(dfbmcut, "simplified_key", 400)
print(g2[['count', 'Percentage']].to_markdown())
print()
print(g2[['count', 'Percentage']].to_latex(float_format="%.2f" ))
categ_before_bysep_cut = g2

| Category           |   count |   Percentage |
|:-------------------|--------:|-------------:|
| Double Charm       |  452226 |   47.1497    |
| Normalization like |  332365 |   34.6528    |
| Bad Xc             |  157220 |   16.392     |
| Signal             |    9961 |    1.03855   |
| Combinatorial      |    5503 |    0.57375   |
| Tau from charm     |    1438 |    0.149928  |
| others             |     415 |    0.0432685 |

\begin{tabular}{lrr}
\toprule
{} &   count &  Percentage \\
Category           &         &             \\
\midrule
Double Charm       &  452226 &       47.15 \\
Normalization like &  332365 &       34.65 \\
Bad Xc             &  157220 &       16.39 \\
Signal             &    9961 &        1.04 \\
Combinatorial      &    5503 &        0.57 \\
Tau from charm     &    1438 &        0.15 \\
others             &     415 &        0.04 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_266248/1558248846.py:4: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(g2[['count', 'Percentage']].to_latex(float_format="%.2f" ))


In [22]:
_, g2 = categ_groupby(dfbmcut, "key", 400)
print(g2[['count', 'Percentage']].to_markdown())
print()
print(g2[['count', 'Percentage']].to_latex(float_format="%.2f", escape=True))

| Category                                         |   count |   Percentage |
|:-------------------------------------------------|--------:|-------------:|
| Xc_signal_Ypis_B_vertex_fromBs                   |  285302 |   29.746     |
| Xc_background                                    |  157220 |   16.392     |
| Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB   |  130226 |   13.5775    |
| Xc_signal_Ypis_displaced_fromBs_fromDs           |   78099 |    8.14271   |
| Xc_signal_Ypis_displaced_fromB0_fromDp           |   65190 |    6.7968    |
| Xc_signal_Ypis_nomatch_doubleCharm               |   38950 |    4.06098   |
| Xc_signal_Ypis_displaced_fromBp_fromD0           |   30025 |    3.13045   |
| Xc_signal_Ypis_nomatch_Prompt                    |   26858 |    2.80025   |
| Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB   |   23271 |    2.42627   |
| Xc_signal_Ypis_displaced_fromLambdab_fromLambdac |   22561 |    2.35224   |
| Xc_signal_Ypis_displaced_fromBs_fromDp           |   16615 |  

/tmp/ipykernel_266248/3451946249.py:4: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(g2[['count', 'Percentage']].to_latex(float_format="%.2f", escape=True))


## Effect of the cut on B_Y_SEP

In [26]:
dfbysep = dfbmcut.query("B_Y_SEP < -4.5")

In [24]:
_, g2 = categ_groupby(dfbysep, "simplified_key", 400)
print(g2[['count', 'Percentage']].to_markdown())
print()
print(g2[['count', 'Percentage']].to_latex(float_format="%.2f" ))
categ_after_bysep_cut = g2

| Category           |   count |   Percentage |
|:-------------------|--------:|-------------:|
| Double Charm       |  209787 |    87.4995   |
| Bad Xc             |   20923 |     8.72672  |
| Signal             |    3594 |     1.49901  |
| Normalization like |    2090 |     0.871712 |
| Combinatorial      |    1985 |     0.827918 |
| Tau from charm     |    1150 |     0.47965  |
| others             |     229 |     0.095513 |

\begin{tabular}{lrr}
\toprule
{} &   count &  Percentage \\
Category           &         &             \\
\midrule
Double Charm       &  209787 &       87.50 \\
Bad Xc             &   20923 &        8.73 \\
Signal             &    3594 &        1.50 \\
Normalization like &    2090 &        0.87 \\
Combinatorial      &    1985 &        0.83 \\
Tau from charm     &    1150 &        0.48 \\
others             &     229 &        0.10 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_266248/1006101757.py:4: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(g2[['count', 'Percentage']].to_latex(float_format="%.2f" ))


In [12]:
_, g2 = categ_groupby(dfbysep, "key", 400)
print(g2[['count', 'Percentage']].to_markdown())
print()
print(g2[['count', 'Percentage']].to_latex(float_format="%.2f", escape=True ))

| Category                                         |   count |   Percentage |
|:-------------------------------------------------|--------:|-------------:|
| Xc_signal_Ypis_displaced_fromBs_fromDs           |   48572 |    20.2588   |
| Xc_signal_Ypis_displaced_fromB0_fromDp           |   48126 |    20.0727   |
| Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB   |   35855 |    14.9547   |
| Xc_background                                    |   20923 |     8.72672  |
| Xc_signal_Ypis_displaced_fromBp_fromD0           |   19002 |     7.92549  |
| Xc_signal_Ypis_nomatch_doubleCharm               |   16249 |     6.77725  |
| Xc_signal_Ypis_displaced_fromBs_fromDp           |   10955 |     4.56919  |
| Xc_signal_Ypis_displaced_fromLambdab_fromLambdac |    6516 |     2.71774  |
| Xc_signal_Ypis_displaced_fromBp_fromDp           |    5973 |     2.49126  |
| Xc_signal_Ypis_diffVertex_doubleCharm            |    4751 |     1.98158  |
| Xc_signal_Ypis_displaced_fromB0_fromD0           |    3924 |  

# Comparison before/after B_Y_SEP

In [13]:
categ_before_bysep_cut

,count,Percentage,cumulative %
Category,,,
Double Charm,452226,47.15,47.15
Normalization like,332365,34.65,81.80
Bad Xc,157220,16.39,98.19
Signal,9961,1.04,99.23
Combinatorial,5503,0.57,99.81
Tau from charm,1438,0.15,99.96
others,415,0.04,100.00


In [14]:
categ_after_bysep_cut

,count,Percentage,cumulative %
Category,,,
Double Charm,209787,87.50,87.50
Bad Xc,20923,8.73,96.23
Signal,3594,1.50,97.73
Normalization like,2090,0.87,98.60
Combinatorial,1985,0.83,99.42
Tau from charm,1150,0.48,99.90
others,229,0.10,100.00


In [15]:
categ_after_bysep_cut["before"] = categ_after_bysep_cut.apply(lambda row: categ_before_bysep_cut.loc[row.name]['count'], axis=1)

In [16]:
categ_after_bysep_cut["cut efficiency"] = categ_after_bysep_cut["count"] / categ_after_bysep_cut["before"]

In [17]:
categ_after_bysep_cut

,count,Percentage,cumulative %,before,cut efficiency
Category,,,,,
Double Charm,209787,87.50,87.50,452226.0,4.64e-01
Bad Xc,20923,8.73,96.23,157220.0,1.33e-01
Signal,3594,1.50,97.73,9961.0,3.61e-01
Normalization like,2090,0.87,98.60,332365.0,6.29e-03
Combinatorial,1985,0.83,99.42,5503.0,3.61e-01
Tau from charm,1150,0.48,99.90,1438.0,8.00e-01
others,229,0.10,100.00,415.0,5.52e-01


In [18]:
print(categ_after_bysep_cut[['count', 'Percentage', 'cut efficiency']].to_latex(float_format="%.3f" ))

\begin{tabular}{lrrr}
\toprule
{} &   count &  Percentage &  cut efficiency \\
Category           &         &             &                 \\
\midrule
Double Charm       &  209787 &      87.499 &           0.464 \\
Bad Xc             &   20923 &       8.727 &           0.133 \\
Signal             &    3594 &       1.499 &           0.361 \\
Normalization like &    2090 &       0.872 &           0.006 \\
Combinatorial      &    1985 &       0.828 &           0.361 \\
Tau from charm     &    1150 &       0.480 &           0.800 \\
others             &     229 &       0.096 &           0.552 \\
\bottomrule
\end{tabular}



## Loading the trained BDT to separate double charm

In [19]:
import joblib
bdt_all = joblib.load("/media/lben/Samsung_T7/data/eos/lhcb/wg/semileptonic/RDsHad/AP/final/train_bdt/output2/bdt_all_200.pkl")

In [20]:
train_columns = [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_BPVVDR",
    "B_M",
    "B_correctedMass",
    "log(abs(PBsn))",
    "log(abs(PBv/B_P))",
    "log(abs(PBvn/B_P))",
    "log(abs((PBsn-PBvn)/PBvn))",
    "log(sqrt(abs(mDs2vn)))",
    "mN2v",
    "log(Y_PE)",
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass",
]

In [21]:
rdf2 = ROOT.RDataFrame("DecayTree", "temp_withbdtallcols.root")
df2 = pd.DataFrame(rdf2.Cache().AsNumpy())

In [22]:
df2.columns

Index(['category', 'simplified', 'B_M', 'B_Y_SEP',
       'Xc_signal_Ypis_displaced_fromBs_fromTau', 'fromY_from_B_vertex',
       'Y_0_40_nc_mult', 'Y_0_20_cc_mult', 'Y_0_20_cc_PZ', 'Y_0_30_nc_PZ',
       'Y_0_40_nc_PZ', 'min_m2pi', 'max_m2pi', 'missing_mass_2', 'BDT_Iso',
       'B_pT_Bdir', 'Y_BPVVDR', 'missing_pY_mass', 'Y_correctedMass', 'q2_2',
       'tauY_2', 'B_BPVVDR', 'mN2v', 'B_correctedMass', 'eventIndex', 'PBsn',
       'PBv', 'PBvn', 'B_P', 'mDs2vn', 'Y_PE'],
      dtype='object')

In [23]:
df3=df2.query("B_M < 5000 & B_Y_SEP < -4.5").copy()

In [24]:
def add_cols_for_bdt(tmpdf):
    df = tmpdf.copy()
    df["log(abs(PBsn))"] = np.log(np.abs(df["PBsn"]))
    df["log(abs(PBv/B_P))"] = np.log(np.abs(df["PBv"] / df["B_P"]))
    df["log(abs(PBvn/B_P))"] = np.log(np.abs(df["PBvn"] / df["B_P"]))
    df["log(abs((PBsn-PBvn)/PBvn))"] = np.log(np.abs((df["PBsn"] - df["PBvn"]) / df["PBvn"]))
    df["log(sqrt(abs(mDs2vn)))"] = np.log(np.sqrt(np.abs(df["mDs2vn"])))
    df["log(Y_PE)"] = np.log(df["Y_PE"]) 
    df["diff_m2pi"] = df["max_m2pi"] - df["min_m2pi"]
    return df
df4 = add_cols_for_bdt(df3)

In [25]:
df4.shape

(239758, 38)

In [26]:
df4['bdt_all'] = bdt_all.predict_proba(df4[train_columns])[:,1]

In [27]:
import categories4 as f
df4['simplified_key'] = df4.apply(lambda row: f.pretty_categories_map[row["simplified"]], axis=1)
df4['key'] = df4.apply(lambda row: f.categories_map[row["category"]], axis=1)

In [28]:
df5 = df4.query("bdt_all > 0.35").copy()
_, g2 = categ_groupby(df5, "simplified_key", 100)
print(g2[['count', 'Percentage']].to_latex())

\begin{tabular}{lrr}
\toprule
{} &  count &  Percentage \\
Category           &        &             \\
\midrule
Double Charm       &  56313 &       83.79 \\
Bad Xc             &   6225 &        9.26 \\
Signal             &   3001 &        4.47 \\
Normalization like &    715 &        1.06 \\
Tau from charm     &    677 &        1.01 \\
Combinatorial      &    234 &        0.35 \\
others             &     45 &        0.07 \\
\bottomrule
\end{tabular}



In [29]:
df5 = df4.query("bdt_all > 0.75").copy()
_, g2 = categ_groupby(df5, "simplified_key", 100)
print(g2[['count', 'Percentage']].to_latex())

\begin{tabular}{lrr}
\toprule
{} &  count &  Percentage \\
Category           &        &             \\
\midrule
Double Charm       &  10282 &       74.14 \\
Signal             &   1693 &       12.21 \\
Bad Xc             &   1432 &       10.33 \\
Normalization like &    236 &        1.70 \\
Tau from charm     &    176 &        1.27 \\
others             &     50 &        0.36 \\
\bottomrule
\end{tabular}



In [31]:
df5 = df4.query("bdt_all > 0.5").copy()
_, g2 = categ_groupby(df5, "simplified_key", 100)
print(g2[['count', 'Percentage']].to_latex())

\begin{tabular}{lrr}
\toprule
{} &  count &  Percentage \\
Category           &        &             \\
\midrule
Double Charm       &  34696 &       81.45 \\
Bad Xc             &   4075 &        9.57 \\
Signal             &   2665 &        6.26 \\
Normalization like &    520 &        1.22 \\
Tau from charm     &    483 &        1.13 \\
Combinatorial      &    129 &        0.30 \\
others             &     29 &        0.07 \\
\bottomrule
\end{tabular}



# Defining the columns in ROOT

In [43]:
rdf10 = ROOT.RDataFrame("DecayTree", "temp_withbdtallcols.root")
rdf10 = rdf10.Define("log_abs_PBsn", "log(abs(PBsn))")
rdf10 = rdf10.Define("log_abs_PBv_B_P", "log(abs(PBv/B_P))")
rdf10 = rdf10.Define("log_abs_PBvn_B_P", "log(abs(PBvn/B_P))")
rdf10 = rdf10.Define("log_abs__PBsn_PBvn_PBvn", "log(abs((PBsn - PBvn) / PBvn))")
rdf10 = rdf10.Define("log_sqrt_abs_mDs2vn", "log(sqrt(abs(mDs2vn)))")
rdf10 = rdf10.Define("log_Y_PE", "log(Y_PE)") 
rdf10 = rdf10.Define("diff_m2pi", "max_m2pi - min_m2pi")